In [1]:
import serial
import numpy as np
from datetime import datetime
import os
import time
from pytz import timezone
import math
import json
import binascii

In [2]:
with open(os.path.join("config_file", "Heat_stage.json"), 'r') as r:
    heater_list = json.load(r)

In [67]:
import serial
import numpy as np
from datetime import datetime
import os
import time
from pytz import timezone
import math
import json
def get_time():
    time_now = timezone('US/Pacific')
    time = str(datetime.now(time_now))[0:16] + "\n"
    return time


class syringe_pump():
    def __init__(self, config):
        self.config = config[0]
        self.com = self.config['port']
        self.dictionary = self.config['Dict']
        self.speed_dict = self.config['speed_code']
        self.syringe_vol = self.config['volume']
        self.buffer_time = 3

    def connect_pump(self):
        self.pump = serial.Serial(
            port=self.com,
            baudrate=9600,
            bytesize=8,
            parity=serial.PARITY_NONE,
            stopbits=serial.STOPBITS_ONE,
            timeout=2
        )

    def disconnect_pump(self):
        self.pump.close()

    def config_pump(self):
        self.connect_pump()
        command = '/1ZR\r'
        self.pump.write(command.encode())
        self.disconnect_pump()

    def set_speed(self, speed_code):
        command = '/1S' + str(speed_code) + 'R\r'
        self.pump.write(command.encode())

    def set_path(self, port):
        command = '/1I' + str(port) + 'R\r'
        self.pump.write(command.encode())

    def fillSyringe(self, vol, speed_code, reagent):
        port = self.dictionary[reagent]
        self.set_path(port)
        time.sleep(2)
        self.set_speed(speed_code)
        time.sleep(0.5)
        increments = str(int(math.ceil((vol / self.syringe_vol) * 181490)))
        command = '/1A' + increments + 'R\r'
        self.pump.write(command.encode())
        timeneed = self.calculate_time(vol, speed_code)
        time.sleep(timeneed + self.buffer_time)
        return timeneed

    def dispense_pump(self, vol, speed_code, path):
        port = self.dictionary[path]
        self.set_path(port)
        time.sleep(2)
        self.set_speed(speed_code)
        time.sleep(0.5)
        increments = str(int(math.ceil((vol / self.syringe_vol) * 181490)))
        command = '/1D' + increments + 'R\r'
        self.pump.write(command.encode())
        timeneed = self.calculate_time(vol, speed_code)
        time.sleep(timeneed + self.buffer_time)
        return timeneed

    def stop_pump(self):
        command = '/1T\r'
        self.pump.write(command.encode())

    def clear_pump_syringe(self):
        self.set_path(5)
        time.sleep(2)
        command = '/1A0R\r'
        self.pump.write(command.encode())
        time.sleep(2)
        self.fillSyringe(self.syringe_vol, 7, "water")
        time.sleep(self.buffer_time)
        self.dispense_pump(self.syringe_vol, 7, "waste")

    def calculate_time(self, vol, speed_code):
        increment_per_seconds = self.speed_dict[str(speed_code)]
        increments = int(math.ceil((vol / self.syringe_vol) * 181490))
        time_use = increments / increment_per_seconds
        return time_use

    def check_status(self):
        command = '/1Q\r'
        self.pump.write(command.encode())
        a = self.pump.readline().strip()
        if str(a)[8] == '@':
            return 0
        if str(a)[8] == "'":
            return 1

    def isPumpRunning(self):
        self.pump.status=0
        return self.pump.status
 
        
    
        

In [1]:
from device.Syringe_pump import Syringe_Pump
import json
import os
import time
import serial
import binascii

In [2]:
ERROR_DESCRIPTIONS = {    0: "No error",    
                      1: "Initialization error",    
                      2: "Invalid command",    
                      3: "Invalid operand",    
                      4: "Invalid command sequence",    
                      5: "EEPROM failure",    
                      6: "Out of range parameter",    
                      7: "Power failure",    
                      8: "Valve response failure",    
                      9: "Plunger response failure",    
                      10: "Plunger blocked",    
                      11: "Valve blocked",    
                      12: "Command overrun"}

In [3]:
with open(os.path.join("config_file", "Tecan_syringe_pump.json"), 'r') as r:
     pump_cfg = json.load(r)

In [4]:
pump=Syringe_Pump(pump_cfg)

In [5]:
pump.connect_pump()
pump.config_pump()

move the pump port to 4


In [10]:
pump.disconnect_pump()

In [14]:
pump.get_current_increment()


b'\xff/0`45158\x03\r\n'


In [7]:
pump.config_pump()

In [6]:
amount=4000
speed=23

In [7]:

pump.set_path("IRM")
time.sleep(1)
pump.fillSyringe(amount,speed)



move the pump port to 8
12500
4000
58077
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1


In [8]:
import math
str(int(math.ceil((4000 / 12500) * 181490)))

'58077'

In [9]:
str(int(math.ceil((2000 / 12500) * 181490)))

'29039'

In [9]:
pump.set_path("chamber1")
time.sleep(1)
pump.dispense_pump(2000,speed)


move the pump port to 1
12500
2000
29038
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1


In [45]:
time.sleep(2)
command = f"/1Q\r" 
ready_bit=0
while ready_bit==0: 
    pump.pump.write(command.encode())
    time.sleep(0.1)
    
    if pump.pump.in_waiting > 0:
        response = pump.pump.read(pump.pump.in_waiting)
        
        # Look for status byte (bits 7,6,4 = 0,1,0)
        for byte in response:
            if (byte & 0b11010000) == 0b01000000:
                ready_bit = (byte >> 5) & 0x01
                error_code = byte & 0x0F
                print(ready_bit)

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1


In [ ]:
while True:        
    pump.pump.reset_input_buffer()       
    command = f"/1Q\r"        
    print(f"Sending command: {command.strip()}")        
    pump.pump.write(command.encode())        
    time.sleep(0.1)       
    if pump.pump.in_waiting > 0:          
        response = pump.pump.read(pump.pump.in_waiting)          
        print(f"Raw response: {response}")            
        print(f"Hex: {binascii.hexlify(response)}")            
        print(f"ASCII: {' '.join([chr(b) if 32 <= b <= 126 else '.' for b in response])}")
 

In [39]:
pump.pump.reset_input_buffer()
command = f"/1Q\r"    
while ready_bit==0: 
    pump.pump.write(command.encode())
    time.sleep(0.1)
    
    if pump.pump.in_waiting > 0:
        response = pump.pump.read(pump.pump.in_waiting)
        
        # Look for status byte (bits 7,6,4 = 0,1,0)
        for byte in response:
            if (byte & 0b11010000) == 0b01000000:
                ready_bit = (byte >> 5) & 0x01
                error_code = byte & 0x0F
                print(ready_bit)
      

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1


KeyboardInterrupt: 

In [10]:
a=pump.check_status()
a

'`'

In [9]:
pump.set_path("batch_reagents")
time.sleep(1)
pump.dispense_pump(1000,24)
time1=pump.calculate_time(1000,24)
time.sleep(time1)

In [41]:
pump.isPumpRunning()

True

In [42]:
pump.set_path("pbst")
time.sleep(1)
pump.dispense_pump(5000,24)
time1=pump.calculate_time(5000,24)
time.sleep(time1)

In [44]:
pump.isPumpRunning()

False

In [38]:

pump.set_path("IRM")
time.sleep(1)
pump.fillSyringe(5000,24)


In [10]:
pump.clear_pump_syringe()

In [11]:
pump.disconnect_pump()